# Week 8 Post-Class Exercise: CIFAR-10 — Solutions

Reference solution. Compare your work to this AFTER you've made an honest attempt.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
import numpy as np
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.flatten()
y_test = y_test.flatten()

cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## Step 3 solution: VGG-style CIFAR-10 CNN

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Step 4 solution: train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=4, restore_best_weights=True, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    epochs=20, batch_size=128, validation_split=0.1,
    callbacks=callbacks, verbose=2
)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\nTest accuracy: {test_acc:.4f}')
# Expected: ~75-78% with this architecture (no augmentation, no batch norm)

## Steps 6-7: training curves and misclassifications

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
predictions = model.predict(X_test, verbose=0)
predicted_labels = np.argmax(predictions, axis=1)
wrong = np.where(predicted_labels != y_test)[0]
print(f'Misclassified: {len(wrong)} of {len(y_test)} ({len(wrong)/len(y_test)*100:.1f}%)')

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i, ax in enumerate(axes):
    idx = wrong[i]
    ax.imshow(X_test[idx])
    p = cifar_classes[predicted_labels[idx]]
    a = cifar_classes[y_test[idx]]
    ax.set_title(f'P:{p}\nA:{a}', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

## Reflection answers (typical)

1. **CIFAR-10 accuracy:** typically 75-78% with this architecture. Compare to ~92% on Fashion-MNIST.

2. **Why CIFAR-10 is harder:**
   - 32×32 RGB has more visual complexity than 28×28 grayscale
   - Intra-class variation: a cat in different poses, colors, backgrounds is very different from another cat
   - Inter-class similarity: cat vs dog, deer vs horse, automobile vs truck — visually overlapping
   - Fashion-MNIST has cleaner backgrounds and more standardized poses

3. **Overfitting check:** Look for training accuracy continuing to climb while validation accuracy plateaus or drops. With this architecture and Dropout, you typically see mild overfitting toward the end. Expected — that's why we use EarlyStopping.

4. **Single architecture change:** Most-impactful would be **data augmentation** (random flips, crops, brightness shifts) — easily +5-10% accuracy. Secondary: **BatchNormalization** between Conv2D layers (faster training, slightly better accuracy).

5. **Foreshadow:** With only 200 images per class, this network would massively overfit and probably hit ~30-40% test accuracy. Transfer learning (Week 9) is the answer.